# Voit-on venir la faillite ?

**Projet du cours « Python pour la data science » (ENSAE, 2A)** – Lucas Poignant, Olivier Caffier, Claudia Aguado

**Problématique.** Les entreprises défaillantes présentent-elles des signaux financiers dégradés dans les années
qui précèdent leur défaillance, et ces signaux sont-ils assez nets pour être repérés à l'avance ?

Ce notebook ne contient que l'argumentation, les tableaux et les graphiques : tous les calculs sont faits par
les fonctions du dossier `src/`. Il relit les résultats intermédiaires enregistrés sur le stockage S3 du SSP Cloud
(lecture publique, sans clé). La chaîne complète, depuis les données brutes, est décrite dans le `README.md`.

In [ ]:
# Fonctions du projet (dossier src/) et réglages d'affichage
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

import config
from src.analyse import graphiques, heterogeneite, portrait, regles_alerte, silence_comptable, temoins, trajectoires
from src.collecte import contexte
from src.nettoyage import base_analyse, nettoyage_bodacc

pd.set_option("display.max_columns", 30)

## Introduction

In [ ]:
df_serie = contexte.lire_serie_defaillances()
fig, ax = graphiques.tracer_courbe(
    df_serie, "date", "nombre_defaillances",
    "Défaillances d'entreprises en France, par trimestre",
    "Nombre de défaillances (données brutes)",
    f"INSEE, série {config.IDBANK_DEFAILLANCES} (par date de jugement)",
)
debut_covid = pd.Timestamp(f"{config.ANNEE_DEBUT_JUGEMENTS}-01-01")
fin_covid = pd.Timestamp(f"{config.ANNEE_FIN_PERIODE_COVID}-12-31")
graphiques.surligner_periode(ax, debut_covid, fin_covid, "période Covid")

> **À rédiger par Claudia.** Points à couvrir :
> - le contexte : l'évolution des défaillances, le creux de la période Covid et le rattrapage ensuite ;
> - l'enjeu : pourquoi repérer tôt une entreprise en difficulté ;
> - la problématique et le plan du rapport.

## Partie 1 – Les données

Le projet croise trois sources publiques, reliées entre elles par le numéro SIREN de l'entreprise.

| Source | Accès | Ce qu'elle apporte |
|---|---|---|
| **BODACC** (DILA) | API OpenDataSoft, export parquet année par année | les jugements d'ouverture de procédure collective (sauvegarde, redressement, liquidation) : *qui* fait défaut et *quand* |
| **Bilans INPI** (données « Signaux Faibles ») | fichier parquet, data.gouv.fr | les postes de la liasse fiscale (chiffre d'affaires, résultat, capitaux propres, dettes, total du bilan…) à partir desquels on calcule les ratios financiers |
| **SIRENE** (INSEE) | stocks parquet des unités légales et des établissements, data.gouv.fr | les caractéristiques de chaque entreprise : activité (section NAF), date de création, effectif, catégorie juridique, département du siège |

Le BODACC fournit l'événement à expliquer, les bilans fournissent les signaux observés avant cet événement, et SIRENE permet de décrire les entreprises et de les comparer à des entreprises semblables (appariement du 1.4). La série trimestrielle des défaillances de l'INSEE, utilisée en introduction, sert seulement de point de comparaison pour vérifier que nos jugements BODACC sont cohérents avec les chiffres officiels (1.1).

**Période d'étude.** On étudie les entreprises dont le premier jugement d'ouverture date de **2020 à 2024**. Les données sont collectées sur une période plus large : le BODACC de 2017 à 2025, pour repérer les entreprises déjà en procédure avant 2020 et pour récupérer les jugements de fin 2024 publiés début 2025 ; les bilans clos de 2015 à 2025, pour disposer de quatre années de comptes (N-4) avant un jugement de début 2020. On s'arrête aux jugements de 2024 car les comptes et les annonces de 2025 sont encore incomplets.

**La période Covid.** Les jugements de 2020 et 2021 sont atypiques : les aides publiques (prêts garantis par l'État, chômage partiel, reports de charges) ont fortement réduit le nombre de défaillances, qui ont ensuite rattrapé leur niveau d'avant-crise. Plutôt que de retirer ces deux années, on les garde et on ajoute une colonne `periode` qui distingue la période **Covid (2020-2021)** de l'**après-Covid (2022-2024)**. Les principaux graphiques sont tracés séparément pour les deux périodes, ce qui permet de vérifier si les signaux d'alerte sont les mêmes quand les défaillances sont retardées par les aides.

### 1.1 Les défaillances (BODACC) comparées à la série officielle

In [ ]:
df_jugements = portrait.lire_jugements_periode()
display(nettoyage_bodacc.comparer_avec_insee(df_jugements))

> **À rédiger par Lucas.** Points à couvrir :
> - le choix du premier jugement d'ouverture par SIREN et les expressions régulières utilisées ;
> - l'écart entre nos jugements et la série INSEE.

### 1.2 Les choix de nettoyage des bilans

Les bilans INPI sont nettoyés en une suite d'étapes (`src/nettoyage/ratios.py`). Chaque choix est noté ci-dessous avec sa raison.

- **Types de bilan gardés : complets (C) et simplifiés (S).** Les bilans consolidés (K) décrivent un groupe et non une unité légale : ils ne se relient pas proprement à un SIREN jugé au BODACC. Les bilans simplifiés sont gardés malgré leurs postes moins détaillés, car ils concernent les petites entreprises, qui sont la majorité des défaillantes.
- **Un seul bilan par entreprise et par année de clôture.** En cas de doublon (dépôt rectificatif, exercice décalé), on garde le bilan qui renseigne le plus de postes, puis le plus récent. Sans ce filtre, une même entreprise compterait deux fois à un même horizon.
- **Postes absents mis à 0, seulement dans les comptes publics.** Dans un compte public, un poste de détail absent signifie « rien » (pas d'emprunt, pas de salarié). Dans un compte confidentiel, il signifie « inconnu » : on le laisse vide. Les totaux (total du bilan, capitaux propres…) ne sont jamais complétés : un total absent reste inconnu.
- **Harmonisation C et S.** Les deux liasses n'utilisent pas les mêmes codes (par exemple `FL` et la somme de `210`, `214` et `218` pour le chiffre d'affaires). Une table de correspondance (`config.CORRESPONDANCE_POSTES`) crée un poste commun pour les deux types.
- **Définition des ratios.** Rentabilité nette = résultat net / chiffre d'affaires ; marge d'exploitation = résultat d'exploitation / chiffre d'affaires ; autonomie financière = capitaux propres / total du bilan ; taux d'endettement = dettes totales / total du bilan ; liquidité = disponibilités / total du bilan ; croissance du chiffre d'affaires par rapport à l'exercice précédent. Un ratio est laissé vide quand son dénominateur est nul ou négatif, car il n'a alors pas de sens. La croissance n'est calculée que si l'exercice précédent existe (deux années consécutives).
- **Winsorisation à 1 % et 99 %.** Un ratio dont le dénominateur est proche de zéro peut prendre des valeurs énormes (une perte de 100 k€ pour 1 k€ de chiffre d'affaires donne -100). On ramène les valeurs au-delà des centiles 1 et 99 à ces bornes, calculées sur tous les bilans et toutes les années, au lieu de supprimer les lignes. Comme on raisonne surtout en médianes et en rangs, ce choix change peu les résultats mais stabilise les intervalles interquartiles.

**Limites.**
- **Pas de liquidité pour les bilans simplifiés** : dans le fichier INPI, l'actif des bilans S ne donne que des totaux, sans les disponibilités. La liquidité et l'indicateur de trésorerie quasi nulle ne portent donc que sur les bilans complets, soit des entreprises un peu plus grandes.
- **Comptes confidentiels** : une entreprise peut ne pas publier son compte de résultat. Son bilan reste connu (autonomie, endettement), mais son chiffre d'affaires et son résultat manquent, donc la rentabilité, la marge et la croissance aussi. Les effectifs varient ainsi d'un ratio à l'autre, et la partie 5 montre que ce silence n'est pas aléatoire.

### 1.3 De tous les jugements à la base d'analyse : l'entonnoir

Le tableau suit les entreprises de la première étape de collecte jusqu'à la base d'analyse. La première ligne compte tous les premiers jugements d'ouverture du BODACC collecté de 2017 à 2025 (402 883). On ne garde ensuite que les entreprises jugées de 2020 à 2024, soit **203 679 entreprises** : c'est la base 100 de la colonne `part_periode_pct`.

Les deux filtres suivants retirent très peu d'entreprises : 203 515 sont retrouvées dans SIRENE (99,9 %) et 203 127 ont une section NAF (99,7 %). Ils sont nécessaires car la section NAF sert ensuite à l'appariement et à la partie 4.

La perte principale vient des comptes annuels : seules **61 722 entreprises (30 %)** ont au moins un bilan clos avant leur jugement dans les comptes INPI collectés (exercices 2015 à 2025). Deux raisons l'expliquent. Les entrepreneurs individuels et les micro-entreprises ne sont pas tenus de déposer des comptes, alors qu'ils représentent une part importante des défaillances. Parmi les sociétés, beaucoup de petites structures ne déposent pas leurs comptes malgré l'obligation, la sanction étant rarement appliquée.

Enfin, on exige un bilan clos dans les quatre ans qui précèdent le jugement. L'horizon est glissant : N-1 désigne un bilan clos dans les 12 mois avant le jugement, N-2 entre 12 et 24 mois, etc. Ce choix évite de mélanger, sous une même « année N-1 » civile, des bilans clos un mois et vingt-trois mois avant le jugement. **47 552 entreprises (23 %)** ont au moins un bilan entre N-1 et N-4. Les 14 170 autres avaient un dernier bilan plus ancien : elles avaient cessé de publier bien avant leur jugement, ce qui annonce le silence comptable étudié en partie 5.

La base d'analyse n'est donc pas représentative de l'ensemble des défaillances : elle porte sur les sociétés qui publient leurs comptes, plus grandes et plus structurées que la moyenne. Nos conclusions valent pour ce champ.

In [ ]:
df_defaillantes, df_bilans_defaillantes = temoins.lire_base_defaillantes()
nombre_jugements_total = base_analyse.compter_jugements_total()
display(base_analyse.construire_entonnoir(nombre_jugements_total, df_defaillantes))

### 1.4 Le groupe témoin

**Pourquoi un groupe témoin.** Un ratio qui se dégrade avant le jugement n'est un signal d'alerte que s'il se dégrade moins chez des entreprises comparables qui ne font pas défaut. Sans point de comparaison, on ne peut pas séparer une dégradation propre aux défaillantes d'un choc commun à toutes les entreprises (Covid, hausse des prix et des taux en 2022-2023).

**La méthode.** Pour chaque défaillante, on retient son bilan de référence, le plus récent entre N-1 et N-4. On définit une « case » par trois critères : la section NAF, la tranche de total du bilan (moins de 100 k€, 100 à 500 k€, 500 k€ à 2 M€, plus de 2 M€) et l'année de clôture du bilan de référence. Dans chaque case, on tire au hasard un témoin parmi les entreprises qui n'apparaissent jamais dans le BODACC collecté (2017-2025) et dont la section et le total du bilan sont connus. Ce réservoir compte 5,79 millions de bilans. Les principaux choix sont les suivants :
- **taille mesurée par le total du bilan** : le chiffre d'affaires est masqué dans 35 % des bilans N-1 (comptes confidentiels) alors que le total du bilan est toujours publié, et l'effectif SIRENE est la valeur actuelle, pas celle de l'année du bilan. Les seuils sont des valeurs rondes proches des quantiles du total du bilan des défaillantes à N-1 (médiane d'environ 525 k€). Ils découpent les bilans des défaillantes en tranches de taille comparable (entre 20 % et 34 % à N-1), sauf la dernière, plus petite (18 % à N-1, 12 % sur l'ensemble des horizons) ;
- **même année de clôture** : les deux bilans comparés ont été établis dans la même conjoncture ;
- **un témoin par défaillante**, et une entreprise ne peut être témoin qu'une fois ;
- **tirage reproductible** : le réservoir est trié puis mélangé avec une graine fixée (2026), donc relancer le code redonne les mêmes paires ;
- **date de jugement fictive** : chaque témoin reçoit la date de jugement de sa défaillante. Ses horizons N-1 à N-4 et sa période (Covid ou après Covid) sont calculés exactement comme pour elle.

**Lecture des tableaux.** Le premier tableau montre que l'appariement ne coûte presque rien. Il part de 47 573 défaillantes ayant un bilan entre N-1 et N-4 : c'est 21 de plus que dans l'entonnoir du 1.3, car ce tableau ne retire pas les entreprises absentes de SIRENE. 28 défaillantes sont écartées faute de section NAF ou de total du bilan, et **les 47 545 restantes ont toutes trouvé un témoin** : le réservoir est assez grand pour qu'aucune case ne soit à court de candidats.

Le second tableau compte les bilans de la base appariée (240 404 bilans au total) par période et par horizon. Le fait marquant est le déséquilibre à N-1 : **10 544 bilans de défaillantes contre 33 974 bilans de témoins**, soit environ 22 % des défaillantes contre 71 % des témoins. Autrement dit, les défaillantes ont à N-1 environ trois fois moins de bilans que leurs témoins, et ce rapport est proche dans les deux périodes : 28 % en période Covid (2 606 bilans contre 9 379) et 32 % après (7 938 contre 24 595). À N-4, il remonte à environ 94 %. Les défaillantes cessent donc de publier à l'approche du jugement. Une partie de l'écart est mécanique : un témoin n'est tenu d'avoir un bilan que l'année de référence de sa défaillante, souvent N-2 ou N-3. Ce silence est étudié en partie 5. Il implique dès maintenant que les comparaisons à N-1 de la partie 3 portent sur une minorité de défaillantes, qui n'est peut-être pas représentative.

In [ ]:
df_reference = temoins.choisir_bilans_reference(df_defaillantes, df_bilans_defaillantes)
df_paires = silence_comptable.lire_paires()
display(temoins.construire_entonnoir_appariement(df_defaillantes, df_reference, df_paires))

df_base = trajectoires.lire_base_appariee()
display(temoins.comparer_groupes(df_base))

**Limites de l'appariement.**
- Le total du bilan de la défaillante est mesuré l'année de référence, alors que sa situation peut déjà être dégradée. Le témoin est donc apparié sur une taille déjà affectée. De plus, les tranches sont larges : à l'intérieur d'une case, les deux groupes peuvent avoir des tailles différentes.
- On n'apparie ni sur l'âge, ni sur la forme juridique, ni sur le département. Les écarts de ratios peuvent donc refléter en partie ces différences, en particulier l'âge, qui joue fortement sur le risque de défaillance (partie 2.2).
- Les témoins sont des entreprises qui publient leurs comptes, donc plutôt mieux organisées que la moyenne des entreprises non défaillantes. Certains ont pu cesser leur activité sans procédure collective (dissolution amiable), et ceux appariés à des jugements de 2024 sont observés peu de temps après : quelques-uns feront peut-être défaut après 2025.
- Avec un témoin par défaillante, la base contient 50 % de défaillantes, contre quelques pour mille dans la population réelle. Les taux de fausses alertes de la partie 6 ne sont donc pas transposables tels quels.

## Partie 2 – Portrait des entreprises défaillantes

In [ ]:
# Lit SIRENE en ligne : compter 2 à 5 minutes
df_comptages = portrait.compter_entreprises_et_defaillances()
periode = f"{config.ANNEE_DEBUT_JUGEMENTS}-{config.ANNEE_FIN_JUGEMENTS}"
titre_axe_taux = "Défaillances pour 1 000 sociétés"
source_portrait = "BODACC, INSEE (SIRENE) ; calculs des auteurs"

### 2.1 Selon le secteur

In [ ]:
df_taux_secteurs = portrait.calculer_taux(df_comptages, "libelle_section")
df_taux_secteurs = df_taux_secteurs.sort_values("taux_pour_mille", ascending=False)
fig, ax = graphiques.tracer_barres_horizontales(
    df_taux_secteurs, "libelle_section", "taux_pour_mille",
    f"Taux annuel de défaillance par secteur, {periode}", titre_axe_taux, source_portrait,
)

> **À rédiger par Lucas.** Points à couvrir :
> - commentaire du graphique par secteur.

### 2.2 Selon l'âge

In [ ]:
df_taux_ages = portrait.calculer_taux(df_comptages, "tranche_age")
fig, ax = graphiques.tracer_barres_horizontales(
    df_taux_ages, "tranche_age", "taux_pour_mille",
    f"Taux annuel de défaillance selon l'âge de la société, {periode}", titre_axe_taux, source_portrait,
)

> **À rédiger par Lucas.** Points à couvrir :
> - commentaire du graphique par âge.

### 2.3 Selon la taille

In [ ]:
df_taux_tailles = portrait.calculer_taux(df_comptages, "groupe_effectifs")
fig, ax = graphiques.tracer_barres_horizontales(
    df_taux_tailles, "groupe_effectifs", "taux_pour_mille",
    f"Taux annuel de défaillance selon l'effectif salarié, {periode}", titre_axe_taux, source_portrait,
)

> **À rédiger par Lucas.** Points à couvrir :
> - commentaire du graphique par taille.

### 2.4 Selon le département

In [ ]:
df_taux_departements = portrait.calculer_taux(df_comptages, "departement")
gdf_departements = portrait.preparer_carte(df_taux_departements)
fig, ax = graphiques.tracer_carte(
    gdf_departements, "taux_pour_mille",
    f"Taux annuel de défaillance par département, {periode}", titre_axe_taux, source_portrait,
)

> **À rédiger par Lucas.** Points à couvrir :
> - commentaire de la carte.

### 2.5 Les types de procédure

In [ ]:
fig, ax = graphiques.tracer_barres_empilees(
    portrait.repartir_procedures(df_jugements),
    "Répartition des premiers jugements par type de procédure", "Part des jugements (%)",
    "BODACC ; calculs des auteurs",
)

> **À rédiger par Lucas.** Points à couvrir :
> - commentaire de la répartition des procédures.

## Partie 3 – Les trajectoires des ratios de N-4 à N-1

### 3.1 Défaillantes contre témoins

In [ ]:
figures_trajectoires = trajectoires.tracer_trajectoires_par_periode(
    df_base, config.RATIOS_TRAJECTOIRES, "Trajectoires des ratios"
)

In [ ]:
# Médianes de chaque ratio, par groupe et par horizon (les deux périodes ensemble)
df_quartiles = trajectoires.calculer_quartiles(df_base, config.RATIOS_TRAJECTOIRES, ["groupe"])
display(trajectoires.presenter_medianes(df_quartiles, ["groupe"]).round(3))

**Les défaillantes sont déjà plus fragiles quatre ans avant le jugement.** Dès N-4, leur autonomie financière médiane est deux fois plus faible que celle des témoins (19 % contre 40 % du total du bilan), leur taux d'endettement plus élevé (80 % contre 60 %) et leur liquidité bien plus basse (6,6 % contre 16 %). L'écart de rentabilité existe aussi, mais il est plus faible (1,2 % contre 3,5 %). Les signaux de **bilan** (solvabilité, endettement, trésorerie) sont donc ceux qui distinguent les défaillantes en premier. Comme N-4 est l'horizon le plus lointain de nos données, on ne peut pas dire depuis quand cet écart existe.

**La rentabilité décroche en dernier, mais brutalement.** Chez les défaillantes, la rentabilité nette médiane passe de 1,2 % à N-4 à -0,9 % à N-2, puis à -6,9 % à N-1. La marge d'exploitation suit le même chemin. La perte apparaît donc surtout dans le dernier exercice avant le jugement. Elle détruit ensuite les fonds propres : l'autonomie financière tombe de 9,5 % à N-2 à 1,1 % à N-1 (presque la moitié des défaillantes ont alors des capitaux propres négatifs, voir 3.4). Chez les témoins, les médianes de rentabilité, de solvabilité et de liquidité restent stables de N-4 à N-1. La croissance du chiffre d'affaires est le ratio le moins net : il ne devient négatif qu'à N-1 (-3,6 %).

**Période Covid et après-Covid.** Le profil général est le même dans les deux périodes. Trois différences apparaissent sur les graphiques :
- la liquidité des **témoins** augmente à N-1 pour les jugements de 2020-2021 (de 14 % à 18 %), sans doute grâce aux prêts garantis par l'État et aux reports de charges, alors que celle des défaillantes reste à 3 % : la trésorerie de crise n'a pas atteint, ou n'a pas suffi, aux entreprises qui ont fait défaut ;
- pendant le Covid, le chiffre d'affaires des défaillantes chute à N-1 (-11,7 %) alors que celui des témoins stagne : la baisse d'activité commune à toutes les entreprises a été plus forte chez les défaillantes. Après le Covid, les témoins voient leur chiffre d'affaires croître (+7,5 % à N-1, en partie l'effet de l'inflation), les défaillantes non ;
- l'autonomie financière médiane des défaillantes de la période Covid est négative à N-1 (-0,6 %). Celles qui ont fait défaut malgré les aides étaient donc dans une situation très dégradée.

Pour les jugements de 2020-2021, les horizons N-1 à N-4 couvrent surtout des exercices d'avant la crise : ces bilans décrivent la fragilité avec laquelle les entreprises sont entrées dans la crise plus que l'effet de la crise elle-même.

### 3.2 Vérification : les entreprises présentes aux quatre horizons

In [ ]:
df_base_complete = trajectoires.garder_entreprises_completes(df_base)
figures_completes = trajectoires.tracer_trajectoires_par_periode(
    df_base_complete, config.RATIOS_TRAJECTOIRES, "Trajectoires des ratios, entreprises présentes de N-4 à N-1"
)

La population du 3.1 change d'un horizon à l'autre : on observe 23 453 défaillantes avec une rentabilité connue à N-4, mais seulement 6 638 à N-1. La dégradation pourrait donc venir d'un changement de composition plutôt que d'une évolution des mêmes entreprises. On refait donc les graphiques sur les seules entreprises présentes aux quatre horizons (4 918 défaillantes et 21 785 témoins).

**La lecture du 3.1 est confirmée.** Les mêmes entreprises suivent le même chemin : la rentabilité médiane passe de 0,9 % à N-4 à -7,4 % à N-1, l'autonomie financière de 20 % à 3 %, et l'écart avec les témoins est le même dès N-4. La dégradation est bien une trajectoire réelle des entreprises, pas un artefact de composition.

**Une nuance** : ces défaillantes « complètes » sont un peu moins dégradées à N-1 que l'ensemble (autonomie de 3,1 % contre 1,1 %, endettement de 96 % contre 98 %). C'est attendu : publier ses comptes chaque année jusqu'au bout est le signe d'une entreprise mieux organisée. C'est un premier indice du biais de sélection étudié en partie 5.

### 3.3 Les rangs sectoriels à N-1 et N-2

In [ ]:
df_base_rangs = trajectoires.ajouter_rangs(df_base, trajectoires.lire_rangs_sectoriels())
display(trajectoires.resumer_rangs(df_base_rangs))

Le rang sectoriel situe chaque bilan parmi tous les bilans INPI de la même section NAF et de la même année : 0 pour le plus faible, 100 pour le plus élevé. Il corrige les différences de niveau entre secteurs (un commerce a naturellement moins de fonds propres qu'une holding).

**Les défaillantes sont dans le bas de leur secteur dès N-2, et encore plus bas à N-1.** À N-1, la défaillante médiane a une autonomie financière plus faible que 88 % des entreprises de son secteur (rang 12,2), une rentabilité plus faible que 85 % (rang 14,7), et un endettement plus élevé que 88 % (rang 87,6). Le passage de N-2 à N-1 fait perdre 6 à 8 points de rang sur la rentabilité, l'autonomie et l'endettement. La liquidité (rang 22,6) et surtout la croissance du chiffre d'affaires (rang 32,7) sont moins discriminantes, ce qui rejoint le 3.1.

**Pourquoi les témoins sont proches de 50.** Un témoin est tiré au hasard parmi les entreprises du même secteur et de la même année : c'est une entreprise « ordinaire », qui se situe donc au milieu de la distribution de son secteur. Ils sont un peu au-dessus de 50 en autonomie et en liquidité (52 à 54) et un peu en dessous en endettement (48). En effet, la population de référence contient aussi les futures défaillantes, alors que les témoins n'ont jamais été jugés : ils sont légèrement plus sains que l'entreprise médiane. Ce résultat valide en partie l'appariement : les témoins représentent bien une entreprise typique de leur secteur.

### 3.4 Les situations extrêmes

In [ ]:
display(trajectoires.compter_situations_extremes(df_base, ["groupe"]))

**Capitaux propres négatifs : un écart qui se creuse surtout dans les deux dernières années.** À N-4, 21,6 % des défaillantes ont déjà des capitaux propres négatifs, contre 12,1 % des témoins (près de deux fois plus). Cette part monte à 25,9 % à N-3, 34,2 % à N-2 et 48,5 % à N-1, alors qu'elle reste stable autour de 12 % chez les témoins. Le plus fort saut a lieu entre N-2 et N-1 (+14 points), au moment où la rentabilité devient fortement négative (3.1) : les pertes du dernier exercice font passer les fonds propres sous zéro. À N-1, une défaillante a quatre fois plus de chances qu'un témoin d'avoir des capitaux propres négatifs. Ce signal est aussi le même dans les deux périodes (50,5 % à N-1 pendant le Covid, 47,8 % après).

**Trésorerie quasi nulle (disponibilités inférieures à 2 % du total du bilan, bilans complets seulement) : un signal plus ancien, mais moins net.** 30,7 % des défaillantes sont déjà dans cette situation à N-4, contre 18,7 % des témoins, puis 41,2 % à N-1 contre 17,4 %. L'écart se creuse lentement (de 12 à 24 points). Mais près de 60 % des défaillantes ont encore plus de 2 % de trésorerie à N-1, et un témoin sur six est sous le seuil : ce signal seul produirait beaucoup de fausses alertes (partie 6). Pour les jugements de 2020-2021, la part est déjà élevée dès N-2 (42,5 % contre 32,9 % après le Covid) : ces entreprises manquaient de trésorerie avant même la crise.

## Partie 4 – Des trajectoires différentes selon les entreprises

In [ ]:
df_sous_groupes = heterogeneite.ajouter_sous_groupes(df_base, heterogeneite.construire_sous_groupes())

### 4.1 Selon le grand secteur

In [ ]:
heterogeneite.afficher_sous_groupe(df_sous_groupes, "grand_secteur")

Le graphique et les tableaux donnent trois ratios, un par grande famille (rentabilité nette, autonomie financière, liquidité). Les témoins ont toujours le même secteur que leur défaillante.

**Le même profil dans tous les secteurs.** Dans chacun des dix grands secteurs, les défaillantes sont en dessous des témoins à tous les horizons, et leur rentabilité devient négative entre N-2 et N-1. La dégradation avant défaillance n'est donc pas propre à un secteur.

**Des différences d'intensité.**
- Le **commerce, transports, hébergement-restauration** est le plus grand sous-groupe (2 404 défaillantes avec une rentabilité connue à N-1, 3 823 avec une autonomie connue). Son autonomie médiane devient légèrement négative à N-1 (-1,7 %).
- La **construction** (1 096 défaillantes à N-1) a la rentabilité la moins dégradée à N-1 (-2,9 %) et une autonomie encore positive (3 %, contre 24 % à N-4). Ses défaillances semblent plus soudaines, peut-être liées à des retards de paiement ou à la hausse du coût des matériaux après 2021 plutôt qu'à une lente érosion.
- L'**immobilier** et les **autres services** ont une autonomie fortement négative à N-1 (-11 % et -12 %).
- La **finance et assurance** (431 défaillantes) est un cas à part : les témoins ont une autonomie très élevée (67 %) et une liquidité très faible (moins de 4 %), typique des sociétés holdings. La rentabilité de -25 % à N-1 (comme dans l'information et communication, 221 défaillantes) est sans doute amplifiée par un chiffre d'affaires faible, qui rend le ratio instable.

**Prudence** : l'agriculture (62 défaillantes à N-1), l'administration-enseignement-santé (146) et l'immobilier (157) ont des effectifs faibles, donc des médianes fragiles.

### 4.2 Selon l'âge au jugement

In [ ]:
heterogeneite.afficher_sous_groupe(df_sous_groupes, "classe_age")

L'âge est mesuré au jugement de la défaillante. Les témoins reçoivent la classe d'âge de leur défaillante, mais ils ne sont pas appariés sur l'âge (1.4) : la comparaison porte donc surtout sur les défaillantes entre elles.

**Les jeunes défaillantes partent de plus bas et s'effondrent plus vite.** Pour les entreprises de moins de 3 ans (683 défaillantes avec une rentabilité connue à N-1, 967 avec une autonomie connue), l'autonomie financière passe de 7,9 % à N-3 à -8,1 % à N-1. Les plus de 10 ans (3 188 et 5 197 défaillantes) partent de 24 % à N-4 et gardent encore 4,7 % à N-1. Les entreprises anciennes ont accumulé des fonds propres, qui amortissent les pertes plus longtemps. Il n'y a pas de valeur à N-4 pour les moins de 3 ans : une entreprise aussi jeune ne peut pas avoir de bilan clos quatre ans avant son jugement.

**La rentabilité ne distingue pas les âges à N-1** (de -5,5 % à -8,5 % selon la classe). En revanche, les défaillantes de plus de 10 ans ont déjà une rentabilité faible à N-4 (0,7 % contre 2,4 % pour les 3-5 ans) : leur défaillance ressemble plus à une érosion lente qu'à une rupture. La liquidité est basse et proche pour toutes les classes (autour de 3 % à N-1).

Les signaux sont donc plus faciles à voir tôt chez les entreprises anciennes (plusieurs bilans, érosion progressive) mais plus marqués chez les jeunes, qui ont aussi moins d'historique comptable à observer.

### 4.3 Selon la taille

In [ ]:
heterogeneite.afficher_sous_groupe(df_sous_groupes, "taille_appariement")

La taille est la tranche de total du bilan utilisée pour l'appariement. Témoins et défaillantes sont donc comparables par construction.

**Les petites entreprises perdent leurs fonds propres en un seul exercice.** Dans la tranche de moins de 100 k€ (2 152 défaillantes avec une autonomie connue à N-1), l'autonomie financière médiane passe de 13,6 % à N-4 à -2,2 % à N-2, puis à -47,2 % à N-1. Avec un bilan aussi petit, une seule perte suffit à rendre les capitaux propres fortement négatifs. Dans les tranches plus grandes, la baisse est plus graduelle et l'autonomie reste positive à N-1 (4,6 % pour 500 k€ à 2 M€, 9,5 % au-delà de 2 M€, 1 861 défaillantes).

**Les grandes défaillantes ont pourtant les pertes les plus fortes à N-1** (-9,6 % de rentabilité nette au-delà de 2 M€, 1 316 défaillantes, contre -5,6 % pour 500 k€ à 2 M€). Leur bilan plus solide leur permet d'absorber ces pertes plus longtemps avant le jugement.

**Liquidité** : elle baisse avec la taille dans les deux groupes (les témoins de plus de 2 M€ n'ont que 8,5 % de disponibilités, contre 25 % pour les moins de 100 k€), mais les défaillantes ont environ quatre à six fois moins de trésorerie que leurs témoins dans toutes les tranches. Ce signal est donc relatif à la taille : un seuil unique de trésorerie (comme celui de la partie 6) touche davantage les grandes entreprises.

Limite : la taille est mesurée sur un bilan déjà proche du jugement, donc parfois déjà dégradé (1.4).

### 4.4 Selon le type de procédure

In [ ]:
heterogeneite.afficher_sous_groupe(df_sous_groupes, "type_procedure")

**Les défaillantes qui publient encore leurs comptes ne sont pas représentatives des procédures.** Parmi les jugements de 2020 à 2024, environ 70 % sont des liquidations directes, 25 % des redressements et 3 % des sauvegardes (2.5). Parmi les défaillantes avec une rentabilité connue à N-1, on compte 3 548 redressements, 2 086 liquidations et 1 004 sauvegardes : les liquidations sont largement sous-représentées, car ce sont surtout des entreprises qui ne publient plus de comptes (partie 5).

**La sauvegarde intervient plus tôt dans la dégradation.** La sauvegarde est demandée par le dirigeant avant la cessation des paiements. Les entreprises concernées ont bien une autonomie financière plus élevée à tous les horizons (9 % à N-1, contre -0,3 % pour le redressement et -3 % pour la liquidation, sur 1 665, 5 712 et 3 162 défaillantes). Les trois trajectoires sont pourtant parallèles : elles perdent de 16 à 22 points d'autonomie de N-4 à N-1.

**La rentabilité, elle, ne distingue pas les procédures** : à N-1, elle est proche de -6 % à -7 % dans les trois cas. Ce sont donc les fonds propres, et non les pertes, qui séparent une entreprise encore sauvable d'une entreprise liquidée.

**Un résultat inattendu** : la liquidité médiane à N-1 est plus élevée pour les liquidations (6,4 %, 2 052 défaillantes) que pour les redressements (2,3 %, 4 548). Cela peut venir d'un effet de composition (les liquidations qui publient encore un bilan complet sont peut-être des sociétés de services ou des holdings avec peu d'actifs). Nous ne l'avons pas vérifié ; il faut donc le lire avec prudence.

## Partie 5 – Le silence comptable

In [ ]:
df_toutes_defaillantes = silence_comptable.lire_defaillantes()
display(silence_comptable.decrire_population_totale(df_toutes_defaillantes, df_paires))

In [ ]:
df_grille = silence_comptable.construire_grille(df_paires, df_toutes_defaillantes)
df_grille = silence_comptable.ajouter_statut(df_grille, df_base)
display(silence_comptable.calculer_taux_silence(df_grille, ["groupe"]))
figures_silence = silence_comptable.tracer_silence_par_periode(df_grille)

In [ ]:
# Biais de sélection : ratios à N-2 des défaillantes, selon qu'elles publient encore à N-1
display(silence_comptable.comparer_publiantes_et_silencieuses(df_base))

**Le silence comptable est massif avant la défaillance.** Parmi toutes les défaillantes de la période, 72,5 % (Covid) et 68,7 % (après Covid) n'ont aucun bilan entre N-4 et N-1 : la base appariée ne représente qu'environ un quart des défaillances. Même dans la base appariée, qui ne contient que des entreprises ayant au moins un bilan, la part des comptes absents passe de 32,4 % à N-4 à 49,4 % à N-2 puis 77,8 % à N-1. Chez les témoins, elle reste entre 20 % et 28 %. Elle est plus élevée à N-1 (28,5 %), car les derniers bilans des témoins appariés aux jugements récents n'étaient pas encore déposés lors de la collecte. Les comptes publics des défaillantes passent de 52,5 % à N-4 à 14,4 % à N-1. La part des comptes confidentiels baisse aussi (de 15 % à 8 %) : à l'approche du jugement, les défaillantes ne choisissent pas de cacher leurs comptes, elles cessent de les déposer.

Ne plus déposer ses comptes est donc en soi un signal : c'est le plus fort de tous à N-1, et il est observable sans aucun calcul de ratio. Une limite est qu'une partie des bilans N-1 manquants n'avait peut-être pas encore à être déposée (le délai légal est de sept mois après la clôture).

**Le biais de sélection pour la partie 3.** Les trajectoires de la partie 3 ne portent que sur les défaillantes qui publient encore. À N-1, elles ne sont qu'environ un cinquième de la base appariée et un vingtième de l'ensemble des défaillantes. Si les entreprises qui se taisent sont les plus dégradées, la partie 3 sous-estime la dégradation. On compare donc, à N-2, les défaillantes qui publient encore à N-1 (7 906) et celles qui se taisent ensuite (16 163). Leurs rentabilités sont identiques (-0,9 %), mais les silencieuses ont des fonds propres plus faibles (autonomie de 8,4 % contre 11,6 %) et un endettement plus élevé (91 % contre 88 %). L'écart reste modéré, mais il va dans le sens attendu : **les médianes de la partie 3 donnent une image plutôt optimiste des défaillantes**, et la vraie dégradation à N-1 est sans doute plus forte. Le tableau 3.2 (entreprises présentes à tous les horizons) va dans le même sens. Enfin, ce biais touche sans doute surtout les liquidations, sous-représentées parmi les défaillantes qui publient (4.4).

## Partie 6 – Des règles d'alerte simples

In [ ]:
df_alertes = regles_alerte.ajouter_alertes(df_base.copy())
display(regles_alerte.calculer_couverture(df_alertes))
display(regles_alerte.tableau_performances(df_alertes))
display(regles_alerte.tableau_performances_par_periode(df_alertes))

> **À rédiger par Lucas.** Points à couvrir :
> - les taux de détection et de fausses alertes à N-1 et N-2 ;
> - pourquoi les fausses alertes seraient plus nombreuses en population réelle (critère de validation de l'étape 11).

## Conclusion et limites

> **À rédiger par Claudia.** Points à couvrir :
> - la réponse à la problématique ;
> - les limites : silence comptable et biais de sélection, appariement, valeurs actuelles des effectifs dans SIRENE, période Covid, comptes récents pas encore déposés.